# Create MBIE Who Got Funded Awards

Creates OpenAlex award rows from the New Zealand Ministry of Business, Innovation and Employment (MBIE) official "Who got funded" workbook.

**Data source:** `https://www.mbie.govt.nz/science-and-technology/science-and-innovation/funding-information-and-opportunities/investment-funds/who-got-funded/` -> official monthly XLSX `who-got-funded.xlsx`. The source page states the workbook lists grants for scientific research and associated activities funded by MBIE since 2015, includes active/matured/terminated contracts, and excludes grants made by the Royal Society Te Aparangi, Health Research Council, and Callaghan Innovation.
**S3 location:** `s3a://openalex-ingest/awards/mbie/mbie_projects.parquet`
**OpenAlex funder:** `F4320321983` - Ministry of Business, Innovation and Employment (ROR `https://ror.org/02jtq1b51`, DOI `10.13039/501100003524`), country `NZ`.
**Provenance:** `mbie_who_got_funded`.
**Priority:** 216.

**Schema choices:**
- The workbook has 3,189 source rows. Repeated `Contract ID` values are identical project records split across appropriations, so the local exporter aggregates to 2,823 unique contract awards.
- Stable `funder_award_id = Contract ID`, preserving the citable MBIE grant identifier for WorkAwards matching. Appropriation splits are retained in `appropriations_json`, `amount_components_json`, and `source_composite_ids_json`.
- `amount` sums positive `All years GST excl.` components across appropriation splits; `0`/blank values remain NULL. `currency = NZD` when amount is present.
- `description` comes from `Public Statement` when published.
- MBIE publishes funded organizations, not PI names. `lead_investigator` is therefore org-level: given/family/orcid NULL and `affiliation.name = Organisation`. `affiliation.country` remains NULL because the workbook has no per-recipient country field.
- `start_date` and `end_date` are real day-level workbook dates; start/end years derive from those dates.


## Step 1: Create Raw Table


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mbie_raw
USING delta
AS
SELECT
    *,
    current_timestamp() AS databricks_ingested_at
FROM parquet.`s3a://openalex-ingest/awards/mbie/mbie_projects.parquet`;


In [ ]:
%sql
SELECT COUNT(*) AS total_mbie_contract_awards FROM openalex.awards.mbie_raw;


## Step 1.5: Inspect Raw Data, Money Fields, and Key Uniqueness

Expected source columns: `funder_award_id`, `contract_id`, `display_name`, `description`, `funder_scheme`, `appropriations_json`, `organisation`, `status`, `amount`, `currency`, `amount_components_json`, `annual_amounts_json`, `start_date`, `end_date`, `start_year`, `end_year`, `source_row_count`, `source_page_url`, `source_workbook_url`, `provenance`, `funder_id`, `funder_display_name`, `downloaded_at`.


In [ ]:
%sql
DESCRIBE openalex.awards.mbie_raw;


In [ ]:
%sql
SELECT column_name
FROM openalex.information_schema.columns
WHERE table_schema = 'awards'
  AND table_name = 'mbie_raw'
  AND LOWER(column_name) RLIKE 'amount|currency|gst|annual|appropriation'
ORDER BY column_name;


In [ ]:
%sql
SELECT funder_award_id, display_name, organisation, funder_scheme,
       amount, currency, start_date, end_date, status, source_row_count
FROM openalex.awards.mbie_raw
LIMIT 10;


In [ ]:
%sql
SELECT
    COUNT(*) AS total_rows,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    COUNT(organisation) AS has_organisation,
    COUNT(funder_scheme) AS has_scheme,
    COUNT(TRY_CAST(amount AS DOUBLE)) AS has_amount,
    MIN(TRY_CAST(amount AS DOUBLE)) AS min_amount,
    MAX(TRY_CAST(amount AS DOUBLE)) AS max_amount,
    COUNT(TRY_CAST(start_year AS INT)) AS has_start_year,
    MIN(TRY_CAST(start_year AS INT)) AS min_start_year,
    MAX(TRY_CAST(start_year AS INT)) AS max_start_year,
    COUNT(TRY_CAST(end_year AS INT)) AS has_end_year,
    MIN(TRY_CAST(end_year AS INT)) AS min_end_year,
    MAX(TRY_CAST(end_year AS INT)) AS max_end_year,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    ROUND(try_divide(COUNT(TRY_CAST(amount AS DOUBLE)), COUNT(*)) * 100.0, 1) AS pct_amount
FROM openalex.awards.mbie_raw;


In [ ]:
%sql
SELECT status, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS nzd_total
FROM openalex.awards.mbie_raw
GROUP BY status
ORDER BY awards DESC;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS awards, ROUND(SUM(TRY_CAST(amount AS DOUBLE)), 0) AS nzd_total
FROM openalex.awards.mbie_raw
GROUP BY funder_scheme
ORDER BY awards DESC
LIMIT 25;


## Step 1.6: Fail-Fast - Verify MBIE Funder Exists

F4320321983 is the specific MBIE funder entity. This assertion prevents a silent zero-row transform if the funder dimension is unexpectedly missing.


In [ ]:
%sql
SELECT assert_true(
    COUNT(*) = 1,
    'Missing MBIE funder F4320321983 in openalex.common.funder'
) AS mbie_funder_exists
FROM openalex.common.funder
WHERE funder_id = 4320321983;


In [ ]:
%sql
SELECT funder_id, display_name, ror_id, doi
FROM openalex.common.funder
WHERE funder_id = 4320321983;


## Step 2: Transform to OpenAlex Awards Schema


In [ ]:
%sql
CREATE OR REPLACE TABLE openalex.awards.mbie_awards
USING delta
AS
WITH funder_resolved AS (
    SELECT
        4320321983 AS funder_id,
        COALESCE(MAX(display_name), 'Ministry of Business, Innovation and Employment') AS display_name,
        COALESCE(MAX(ror_id), 'https://ror.org/02jtq1b51') AS ror_id,
        COALESCE(MAX(doi), '10.13039/501100003524') AS doi
    FROM openalex.common.funder
    WHERE funder_id = 4320321983
),
raw_prepped AS (
    SELECT
        *,
        TRY_CAST(amount AS DOUBLE) AS amount_double,
        TRY_TO_DATE(start_date, 'yyyy-MM-dd') AS start_date_parsed,
        TRY_TO_DATE(end_date, 'yyyy-MM-dd') AS end_date_parsed,
        NULLIF(TRIM(organisation), '') AS leader_affiliation_name,
        NULLIF(TRIM(display_name), '') AS display_name_norm,
        NULLIF(TRIM(description), '') AS description_norm,
        NULLIF(TRIM(funder_scheme), '') AS funder_scheme_norm
    FROM openalex.awards.mbie_raw
)
SELECT
    abs(xxhash64(CONCAT('4320321983:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS id,
    rp.display_name_norm AS display_name,
    rp.description_norm AS description,
    4320321983 AS funder_id,
    rp.funder_award_id,
    CASE WHEN rp.amount_double > 0 THEN rp.amount_double ELSE NULL END AS amount,
    CASE WHEN rp.amount_double > 0 THEN 'NZD' ELSE NULL END AS currency,
    struct(
        CONCAT('https://openalex.org/F', TRY_CAST(f.funder_id AS STRING)) AS id,
        f.display_name,
        f.ror_id,
        f.doi
    ) AS funder,
    CASE
        WHEN LOWER(COALESCE(rp.funder_scheme_norm, '')) RLIKE 'fellowship|scholarship|whitinga' THEN 'fellowship'
        WHEN LOWER(COALESCE(rp.funder_scheme_norm, '')) RLIKE 'studentship|internship|training' THEN 'training'
        ELSE 'research'
    END AS funding_type,
    rp.funder_scheme_norm AS funder_scheme,
    'mbie_who_got_funded' AS provenance,
    rp.start_date_parsed AS start_date,
    rp.end_date_parsed AS end_date,
    CASE
        WHEN YEAR(rp.start_date_parsed) > YEAR(current_date()) + 1 THEN NULL
        ELSE YEAR(rp.start_date_parsed)
    END AS start_year,
    CASE
        WHEN YEAR(rp.start_date_parsed) > YEAR(current_date()) + 1 THEN NULL
        ELSE YEAR(rp.end_date_parsed)
    END AS end_year,
    CASE
        WHEN rp.leader_affiliation_name IS NOT NULL THEN
            struct(
                CAST(NULL AS STRING) AS given_name,
                CAST(NULL AS STRING) AS family_name,
                CAST(NULL AS STRING) AS orcid,
                CAST(NULL AS DATE) AS role_start,
                struct(
                    rp.leader_affiliation_name AS name,
                    CAST(NULL AS STRING) AS country,
                    CAST(NULL AS ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>) AS ids
                ) AS affiliation
            )
        ELSE NULL
    END AS lead_investigator,
    CAST(NULL AS STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >) AS co_lead_investigator,
    CAST(NULL AS ARRAY<STRUCT<
        given_name:STRING, family_name:STRING, orcid:STRING, role_start:DATE,
        affiliation:STRUCT<name:STRING, country:STRING, ids:ARRAY<STRUCT<id:STRING, type:STRING, asserted_by:STRING>>>
    >>) AS investigators,
    rp.source_page_url AS landing_page_url,
    CAST(NULL AS STRING) AS doi,
    CONCAT('https://api.openalex.org/works?filter=awards.id:G',
           TRY_CAST(abs(xxhash64(CONCAT('4320321983:', LOWER(TRIM(rp.funder_award_id))))) % 9000000000 AS STRING)) AS works_api_url,
    current_timestamp() AS created_date,
    current_timestamp() AS updated_date
FROM raw_prepped rp
CROSS JOIN funder_resolved f
WHERE rp.funder_award_id IS NOT NULL
  AND rp.display_name_norm IS NOT NULL;


## Step 3: Insert into `openalex_awards_raw` at Priority 216


In [ ]:
%sql
DELETE FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mbie_who_got_funded' AND priority = 216;

INSERT INTO openalex.awards.openalex_awards_raw
SELECT
    id, display_name, description, funder_id, funder_award_id,
    amount, currency, funder, funding_type, funder_scheme, provenance,
    start_date, end_date, start_year, end_year,
    lead_investigator, co_lead_investigator, investigators,
    landing_page_url, doi, works_api_url,
    created_date, updated_date,
    216 AS priority
FROM openalex.awards.mbie_awards;


## Step 6: Verification


In [ ]:
%sql
SELECT COUNT(*) AS total_mbie_awards FROM openalex.awards.mbie_awards;


In [ ]:
%sql
-- Duplicate funder_award_id / id must be zero.
SELECT
    COUNT(*) AS total_rows,
    COUNT(DISTINCT funder_award_id) AS distinct_funder_award_ids,
    COUNT(DISTINCT id) AS distinct_openalex_award_ids,
    COUNT(*) - COUNT(DISTINCT funder_award_id) AS duplicate_funder_award_ids,
    COUNT(*) - COUNT(DISTINCT id) AS duplicate_openalex_ids
FROM openalex.awards.mbie_awards;


In [ ]:
%sql
-- Completeness.
SELECT
    COUNT(*) AS total,
    COUNT(display_name) AS has_title,
    COUNT(description) AS has_description,
    SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END) AS has_positive_amount,
    SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END) AS has_org_level_lead,
    SUM(CASE WHEN lead_investigator.affiliation.country IS NOT NULL THEN 1 ELSE 0 END) AS has_affiliation_country,
    COUNT(start_date) AS has_start_date,
    COUNT(end_date) AS has_end_date,
    COUNT(funder_scheme) AS has_funder_scheme,
    ROUND(try_divide(COUNT(display_name), COUNT(*)) * 100.0, 1) AS pct_title,
    ROUND(try_divide(SUM(CASE WHEN amount > 0 THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_positive_amount,
    ROUND(try_divide(SUM(CASE WHEN lead_investigator.affiliation.name IS NOT NULL THEN 1 ELSE 0 END), COUNT(*)) * 100.0, 1) AS pct_org_level_lead,
    collect_set(currency) AS currencies
FROM openalex.awards.mbie_awards;


In [ ]:
%sql
-- Amount coverage (not waived: MBIE publishes All years GST excl.).
SELECT
    COUNT(*) AS total,
    COUNT(amount) AS has_amount,
    ROUND(try_divide(COUNT(amount), COUNT(*)) * 100.0, 1) AS pct_amount,
    collect_set(currency) AS currencies,
    MIN(amount) AS min_amount,
    MAX(amount) AS max_amount,
    ROUND(AVG(amount), 2) AS avg_amount,
    percentile_approx(amount, 0.5) AS median_amount
FROM openalex.awards.mbie_awards;


In [ ]:
%sql
-- Future-year placeholder guard: starts beyond next calendar year should be nulled.
SELECT COUNT(*) AS future_start_year_rows
FROM openalex.awards.mbie_awards
WHERE start_year > YEAR(current_date()) + 1;


In [ ]:
%sql
SELECT funding_type, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS nzd_total
FROM openalex.awards.mbie_awards
GROUP BY funding_type
ORDER BY awards DESC;


In [ ]:
%sql
SELECT funder_scheme, COUNT(*) AS awards, ROUND(SUM(amount), 0) AS nzd_total
FROM openalex.awards.mbie_awards
GROUP BY funder_scheme
ORDER BY awards DESC
LIMIT 25;


In [ ]:
%sql
SELECT id, funder_award_id, SUBSTRING(display_name, 1, 90) AS title,
       amount, currency, start_date, end_date,
       SUBSTRING(lead_investigator.affiliation.name, 1, 70) AS organisation,
       lead_investigator.affiliation.country AS affiliation_country
FROM openalex.awards.mbie_awards
ORDER BY start_date DESC, funder_award_id
LIMIT 20;


In [ ]:
%sql
-- Confirm the priority insert landed.
SELECT priority, provenance, COUNT(*) AS rows
FROM openalex.awards.openalex_awards_raw
WHERE provenance = 'mbie_who_got_funded'
GROUP BY priority, provenance;


## Handoff / Admin Notes

Contractor validation completed locally with `scripts/local/mbie_to_s3.py --skip-upload`: 2,823 unique contract awards from 3,189 workbook rows; 99.9% positive NZD amount coverage; 100% title, organization, scheme, status, start date, and end date coverage. Contractor has no S3/Databricks access, so admin must upload the parquet, run this notebook, and perform Databricks/API QA.
